# Ranking metrics in NumPy

A wafer inspection camera scores every chip for how likely it is to be defective, and
reports the score to two decimals, so many chips share a score. Before anyone picks a
threshold, the scores can be judged by how they rank the chips. You will compute ROC-AUC
with tied scores, the points of the precision-recall curve, average precision, and the best
precision available when the line must catch 90% of the defective chips.

**How to work through it.** Each part asks for one or two functions. Replace the `TODO` and the
`raise NotImplementedError` with your code, then run the check cell under it. The check
prints `Part N passed` when your function is right, and stops with a message that says
what is wrong when it is not.

## Setup

Run this cell first.

In [ ]:
import numpy as np

# 8,000 chips from one batch: y is 1 for a defective chip, scores are the camera's
# defect scores, rounded to two decimals as the camera reports them.
rng = np.random.default_rng(31)
n = 8_000
y = (rng.random(n) < 0.025).astype(int)
scores = np.round(1 / (1 + np.exp(-(rng.normal(0.0, 1.0, n) + 2.4 * y - 2.5))), 2)
print(f"{n} chips, {y.sum()} defective, {len(np.unique(scores))} distinct scores")

## Part 1 (coding task): ROC-AUC with tied scores

Write `roc_auc(y, s)` for 0/1 labels `y` and scores `s`. Pair every positive with every
negative, and return the share of pairs that the scores put in the right order, counting a
pair with equal scores as one half. That is the area under the ROC curve, ties included, and
the number `roc_auc_score` returns.

Any correct method passes. Comparing every positive with every negative works for the data
here ($214 \times 7{,}786$ pairs). A sorting method scales to millions of rows: sort
the negative scores once, and `np.searchsorted` gives, for every positive at once, how many
negatives score below it and how many score the same.

In [ ]:
def roc_auc(y, s):
    """Fraction of (positive, negative) pairs ordered correctly, a tie counting 1/2."""
    y = np.asarray(y).astype(bool)
    s = np.asarray(s, dtype=float)
    # TODO
    raise NotImplementedError

In [ ]:
def close(a, b):
    return abs(float(a) - b) < 1e-9

assert close(roc_auc([1, 1, 0, 1, 0, 0], [6, 5, 4, 3, 2, 1]), 8 / 9), "no ties: 8 of the 9 pairs are ordered correctly"
assert close(roc_auc([0, 0, 1, 1], [0.1, 0.2, 0.3, 0.4]), 1.0), "a perfect ranking has AUC 1"
assert close(roc_auc([1, 1, 0, 0], [0.1, 0.2, 0.3, 0.4]), 0.0), "a reversed ranking has AUC 0"
assert close(roc_auc([0, 1, 0, 1], [0.5, 0.5, 0.5, 0.5]), 0.5), "equal scores: every pair is a tie, so the AUC is 1/2"
assert close(roc_auc([1, 0, 1, 0, 0, 1, 0, 1], [0.9, 0.8, 0.8, 0.6, 0.5, 0.5, 0.3, 0.1]), 9 / 16), "with ties: each tied pair counts 1/2"
assert close(roc_auc(np.array([True, False, True]), np.array([3, 1, 2])), 1.0), "boolean labels should work too"
i = np.arange(3000)
s_big = (i * 37 % 101) / 100                     # 101 distinct scores, many ties
y_big = (i * 37 % 101) + (i * 13 % 61) > 110
assert close(roc_auc(y_big, s_big), 2717159 / 2951200), "wrong AUC on 3,000 rows with many ties"
print("Part 1 passed")

## Part 2 (coding task): The points of the precision-recall curve

Write `pr_curve(y, s)`. A threshold $t$ flags every example whose score is at least $t$.
Use each distinct score as a threshold, from the highest down, and return three arrays of
equal length: the thresholds, the precision at each, and the recall at each.

For example, the scores `[0.9, 0.8, 0.8, 0.6]` give the thresholds `[0.9, 0.8, 0.6]`, and
the threshold `0.8` flags both examples that score 0.8.

In [ ]:
def pr_curve(y, s):
    """Distinct scores from highest to lowest, with the precision and recall at each."""
    y = np.asarray(y).astype(int)
    s = np.asarray(s, dtype=float)
    # TODO
    raise NotImplementedError

In [ ]:
t, p, r = pr_curve([1, 0, 1, 0, 0, 1, 0, 1], [0.9, 0.8, 0.8, 0.6, 0.5, 0.5, 0.3, 0.1])
assert len(t) == 6 and np.allclose(t, [0.9, 0.8, 0.6, 0.5, 0.3, 0.1]), f"thresholds {t}: one per distinct score, highest first"
assert np.allclose(r, [1 / 4, 2 / 4, 2 / 4, 3 / 4, 3 / 4, 1]), f"recall {r}"
assert np.allclose(p, [1, 2 / 3, 2 / 4, 3 / 6, 3 / 7, 4 / 8]), f"precision {p}: a threshold flags every example tied at it"
t, p, r = pr_curve([0, 1, 1], [1.0, 2.0, 3.0])
assert np.allclose(t, [3, 2, 1]) and np.allclose(p, [1, 1, 2 / 3]) and np.allclose(r, [1 / 2, 1, 1]), "wrong points on an untied example"
print("Part 2 passed")

## Part 3 (coding task): Average precision and a recall target

Write two functions on top of `pr_curve`.

- `average_precision(y, s)` returns $\text{AP} = \sum_n (R_n - R_{n-1})\,P_n$, where
  $P_n$ and $R_n$ are the precision and recall at the $n$-th threshold from Part 2 and
  $R_0 = 0$. This is the definition `average_precision_score` uses.
- `precision_at_recall(y, s, target)` returns the highest precision among the thresholds
  whose recall is at least `target`. A line that must catch that share of the defective
  chips can do no better.

The check then runs all three functions on the chips from the setup.

In [ ]:
def average_precision(y, s):
    # TODO
    raise NotImplementedError


def precision_at_recall(y, s, target):
    # TODO
    raise NotImplementedError

In [ ]:
y8, s8 = [1, 0, 1, 0, 0, 1, 0, 1], [0.9, 0.8, 0.8, 0.6, 0.5, 0.5, 0.3, 0.1]
assert np.isclose(average_precision(y8, s8), 2 / 3), "AP with ties: expected (1 + 2/3 + 1/2 + 1/2) / 4 = 2/3"
assert np.isclose(average_precision([1, 0], [0.5, 0.5]), 0.5), "a tie: both examples are flagged together, so precision is 1/2 when the positive is caught"
assert np.isclose(average_precision([1, 1, 0, 0], [4, 3, 2, 1]), 1.0), "a perfect ranking has AP 1"
yd, sd = [1, 0, 1, 1, 0, 0, 1, 0], [8, 7, 6, 5, 4, 3, 2, 1]
assert np.isclose(average_precision(yd, sd), 251 / 336), "AP = (1 + 2/3 + 3/4 + 4/7) / 4 = 251/336: use each threshold's own precision, not the best precision further down"
assert np.isclose(precision_at_recall(yd, sd, 0.5), 3 / 4), "recall target 0.5: take the best precision over every threshold that reaches it, not the first"
assert np.isclose(precision_at_recall(yd, sd, 1.0), 4 / 7), "recall target 1.0: flagging all eight is not the best way to reach it"
assert np.isclose(precision_at_recall([1, 1, 0, 0], [0.9, 0.6, 0.6, 0.2], 1.0), 2 / 3), "a tie: the threshold 0.6 flags both chips that score 0.6"

auc, ap = roc_auc(y, scores), average_precision(y, scores)
print(f"defective share          {y.mean():.3f}")
print(f"ROC-AUC                  {auc:.3f}   (a random ranker scores 0.5)")
print(f"average precision        {ap:.3f}   (a random ranker scores about {y.mean():.3f})")
print(f"precision at recall 0.9  {precision_at_recall(y, scores, 0.9):.3f}")
assert 0.9 < auc < 1 and ap < auc, "on the chips, expect a high ROC-AUC and a lower AP"
print("Part 3 passed")

## Solution

### Part 1: ROC-AUC with tied scores

In [ ]:
def roc_auc(y, s):
    """Fraction of (positive, negative) pairs ordered correctly, a tie counting 1/2."""
    y = np.asarray(y).astype(bool)
    s = np.asarray(s, dtype=float)
    negatives = np.sort(s[~y])
    positives = s[y]
    below = np.searchsorted(negatives, positives, side="left")              # negatives strictly lower
    level = np.searchsorted(negatives, positives, side="right") - below     # negatives with an equal score
    return (below.sum() + 0.5 * level.sum()) / (len(positives) * len(negatives))

**Why it works.** The AUC counts, for each positive, the negatives below it plus half the negatives level
with it. In the sorted negatives, `searchsorted(..., side="left")` is the number strictly
below a value and `side="right"` the number below or equal, so their difference is the
number of ties. Both run for all positives in one call, and with the sort that makes the
whole function $O(n \log n)$, against $O(n_+ n_-)$ for comparing every pair. The rank-sum
(Mann-Whitney) formula is another route to the same number.

### Part 2: The points of the precision-recall curve

In [ ]:
def pr_curve(y, s):
    """Distinct scores from highest to lowest, with the precision and recall at each."""
    y = np.asarray(y).astype(int)
    s = np.asarray(s, dtype=float)
    order = np.argsort(-s, kind="mergesort")
    s_desc, y_desc = s[order], y[order]
    tp = np.cumsum(y_desc)                       # true positives among the top k
    flagged = np.arange(1, len(s) + 1)           # k
    last = np.r_[np.flatnonzero(np.diff(s_desc)), len(s) - 1]   # last row of each tied block
    return s_desc[last], tp[last] / flagged[last], tp[last] / y.sum()

**Why it works.** No threshold can separate examples with equal scores, so the curve has one point per
distinct score, read at the last example of each tied block. Sorting once and taking
cumulative sums gives the true positives and the number flagged at every cut in a single
pass: precision divides the first by the second, recall divides it by the number of
positives.

### Part 3: Average precision and a recall target

In [ ]:
def average_precision(y, s):
    _, precision, recall = pr_curve(y, s)
    return float(np.sum(np.diff(np.r_[0.0, recall]) * precision))


def precision_at_recall(y, s, target):
    _, precision, recall = pr_curve(y, s)
    return float(precision[recall >= target].max())

**Why it works.** Recall only moves at thresholds that catch a new positive, so the sum gives each positive a
weight of $1/n_+$ times the precision at the threshold that first catches it. Without ties,
AP is the mean over the positives of the precision you have when you reach each one. A
positive tied with negatives gets the precision of the whole tied block. The precision
at a recall target must search every qualifying threshold, because precision is not
monotone in the threshold: in the check, the cut at score 6 reaches recall 1/2 with
precision 2/3, and the lower cut at 5 does better with 3/4.

On the chips the ROC-AUC is about 0.96 and the average precision about 0.60, against 0.5
and about 0.03 for a random ranker. To catch 90% of the defective chips, the best threshold
flags 1,169 chips, 194 of them defective. As an ROC point that is a true positive rate of
0.91 at a false positive rate of 0.125, which looks like a good trade. In chips it is 975
good chips sent for rework, five for every defective chip caught, and the precision of 0.17
is the only one of these numbers that says so.